# 00 — Visão geral

## Por que esta pasta existe

Phase 0 do projeto (replicar SIR / wave-based) está rodada. Antes de pular para a construção do grafo (Phase 1) e o GNN (Phase 2), vale parar e olhar com calma o que o **MGD+** + Kaggle nos entregam — e o que falta. A ideia é sair desta pasta com um esboço sólido de como esses dados viram um **grafo heterogêneo temporal**.

## Mapa dos próximos notebooks

| # | Notebook | Pergunta que responde |
|---|---|---|
| 01 | charts | Como as músicas se comportam no Top 200 e Viral 50? Onde estão os gaps? |
| 02 | songs_features | Que informação acústica/contextual cada música carrega? |
| 03 | artists | Quem são os artistas, quão prolíficos, quão colaborativos? |
| 04 | networks | Como gêneros e artistas se conectam (já temos as redes do MGD+)? |
| 05 | subset_viral_hit | O subset experimental do paper (1.977 → nosso 1.179). Quem entra, quem fica de fora? |
| 06 | gnn_design_sketch | Síntese: nós, arestas, features e janela temporal do grafo heterogêneo. |

## Fontes consideradas

- **MGD+** (Zenodo 8086643, CC-BY-4.0) — `data/MGDplus/`. Top 200 BR completo (2017-01 → 2022-03), features acústicas, redes de gênero e de artistas.
- **Kaggle** `dhruvildave/spotify-charts` (ODbL-1.0) — `data/charts/spotify_charts_br_2017_2021.csv`. Inclui Viral 50 (não está no MGD+), mas com cobertura ~60% por dia.

Decisão prática: usamos MGD+ para tudo que ele cobre; Viral 50 vem do Kaggle, com a limitação documentada.

In [1]:
from pathlib import Path
import pandas as pd

# Caminhos absolutos (a pasta exploration/ fica na raiz do repo)
ROOT = Path.cwd().resolve()
if ROOT.name == "exploration":
    ROOT = ROOT.parent
DATA = ROOT / "data"
MGDPLUS = DATA / "MGDplus"
print("Repo root:", ROOT)
print("data/ existe:", DATA.exists())
print("data/MGDplus/ existe:", MGDPLUS.exists())

Repo root: /home/cristianomendieta/estudos/mestrado/mestrado-roadmap/music-diffusion-gnn
data/ existe: True
data/MGDplus/ existe: True


## Inventário rápido

Para cada fonte: caminho, tamanho aproximado, e o que ela vai virar no grafo.

In [2]:
inventory = [
    {
        "fonte": "Charts BR — Top 200",
        "arquivo": "data/MGDplus/charts/regional/spotify_charts_regional_br.csv",
        "papel no grafo": "sinal temporal (target) por música; arestas music↔chart",
    },
    {
        "fonte": "Charts BR — Viral 50 (+ Top 200 alternativo)",
        "arquivo": "data/charts/spotify_charts_br_2017_2021.csv",
        "papel no grafo": "sinal temporal (target) — único caminho público p/ Viral 50",
    },
    {
        "fonte": "Songs (features acústicas + metadata)",
        "arquivo": "data/songs/br-hit_songs-*.csv",
        "papel no grafo": "atributos do nó music",
    },
    {
        "fonte": "Artists",
        "arquivo": "data/artists/br-artists-all_time.csv",
        "papel no grafo": "atributos do nó artist; ligação artist→music",
    },
    {
        "fonte": "Genre network (co-ocorrência ponderada)",
        "arquivo": "data/MGDplus/genre_network/br/br-genre_network-YYYY.csv",
        "papel no grafo": "arestas genre↔genre (peso=co-ocorrência)",
    },
    {
        "fonte": "Artist network (colaborações)",
        "arquivo": "data/MGDplus/artist_network/br/br-artist_network-YYYY.csv",
        "papel no grafo": "arestas artist↔artist (peso=#colaborações)",
    },
    {
        "fonte": "Subset viral∩hit",
        "arquivo": "data/processed/subset_ids.json",
        "papel no grafo": "conjunto de músicas-foco (treino/avaliação, idem paper)",
    },
]
pd.DataFrame(inventory)

,fonte,arquivo,papel no grafo
0,Charts BR — Top 200,data/MGDplus/charts/regional/spotify_charts_re...,sinal temporal (target) por música; arestas mu...
1,Charts BR — Viral 50 (+ Top 200 alternativo),data/charts/spotify_charts_br_2017_2021.csv,sinal temporal (target) — único caminho públic...
2,Songs (features acústicas + metadata),data/songs/br-hit_songs-*.csv,atributos do nó music
3,Artists,data/artists/br-artists-all_time.csv,atributos do nó artist; ligação artist→music
4,Genre network (co-ocorrência ponderada),data/MGDplus/genre_network/br/br-genre_network...,arestas genre↔genre (peso=co-ocorrência)
5,Artist network (colaborações),data/MGDplus/artist_network/br/br-artist_netwo...,arestas artist↔artist (peso=#colaborações)
6,Subset viral∩hit,data/processed/subset_ids.json,"conjunto de músicas-foco (treino/avaliação, id..."


In [3]:
# Carga rápida de cada fonte para confirmar que está acessível.
# Os notebooks seguintes vão a fundo em cada uma.
import sys
sys.path.insert(0, str(ROOT / "src"))
from music_diffusion_gnn.data.loaders import load_charts, load_songs, load_artists

charts = load_charts()           # auto: MGD+ se data/charts/mgdplus/ existir, senão Kaggle
songs = load_songs()
artists = load_artists()

print(f"charts: {charts.shape[0]:,} linhas, {charts['song_id'].nunique():,} músicas únicas, charts={sorted(charts['chart'].dropna().unique())}")
print(f"songs:  {songs.shape[0]:,} músicas, {songs.shape[1]} colunas")
print(f"artists:{artists.shape[0]:,} artistas, {artists.shape[1]} colunas")

charts: 280,932 linhas, 6,469 músicas únicas, charts=['top200', 'viral50']
songs:  5,010 músicas, 25 colunas
artists:1,701 artistas, 7 colunas


In [4]:
# Subset viral∩hit já materializado em data/processed/
import json
subset_path = DATA / "processed" / "subset_ids.json"
if subset_path.exists():
    payload = json.loads(subset_path.read_text())
    print(f"Subset salvo em {subset_path.name}")
    print(f"  n = {payload['n']:,} músicas")
    print(f"  require_acoustic = {payload['require_acoustic']}")
    print(f"  gerado em {payload['generated_at']}")
else:
    print("Subset ainda não materializado — rodar build_subset() (notebook 05).")

Subset salvo em subset_ids.json
  n = 1,179 músicas
  require_acoustic = False
  gerado em 2026-05-04


## Observações iniciais

1. **Top 200 BR completo** vem do MGD+ (período 2017-01 → 2022-03, 200 músicas/dia).
2. **Viral 50** só existe no Kaggle e tem ~60% de cobertura por dia — limitação a declarar no paper.
3. **Features acústicas** cobrem 100% do Top 200 (mesma origem MGD+).
4. **Redes** já vêm pré-calculadas (gênero ponderada por co-ocorrência, artista ponderada por colaborações). Boa notícia: não vamos reconstruí-las do zero, vamos só carregar.

Os próximos notebooks aprofundam cada bloco e terminam com uma seção **"Insight para o GNN"** que alimenta o esboço final em [06_gnn_design_sketch.ipynb](06_gnn_design_sketch.ipynb).